# 27.5 Toward AGI: open problems

AGI is not one score going up; it is many abilities holding together under shift. This notebook simulates scorecards, robustness, continual learning, and alignment floors without making AGI claims. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(275)


def safe_geomean(x):
    x = np.clip(np.asarray(x, dtype=float), 1e-9, 1.0)
    return float(np.exp(np.mean(np.log(x))))

## The concept, built once (D1)

With capability vector $c=[0.9,0.8,0.2,0.4]$, equal weights give arithmetic average $0.575$, geometric mean $0.490$, and bottleneck $0.200$. These disagree because a weak coordinate cannot be averaged away.

In [ ]:
def agi_score(capabilities, shift_scores=None, floors=None, weights=None):
    capabilities = np.asarray(capabilities, dtype=float)
    if weights is None:
        weights = np.ones_like(capabilities) / len(capabilities)
    average = float(np.dot(weights, capabilities))
    geometric = safe_geomean(capabilities)
    bottleneck = float(np.min(capabilities))
    retained = 1.0
    if shift_scores is not None:
        shift_scores = np.asarray(shift_scores, dtype=float)
        retained = float(np.mean(shift_scores) / max(np.mean(capabilities), 1e-9))
    floor_score = bottleneck
    if floors is not None:
        floors = np.asarray(floors, dtype=float)
        floor_score = float(np.min(capabilities / np.maximum(floors, 1e-9)))
        floor_score = min(floor_score, bottleneck)
    robust = min(geometric * retained, floor_score)
    return {"average": average, "geometric": geometric, "bottleneck": bottleneck, "retained": retained, "floor_score": floor_score, "robust": robust}

lesson_caps = np.array([0.9, 0.8, 0.2, 0.4])
lesson_score = agi_score(lesson_caps)

assert round(lesson_score["average"], 3) == 0.575
assert round(lesson_score["geometric"], 3) == 0.490
assert round(lesson_score["bottleneck"], 3) == 0.200

print(lesson_score)

Robustness and alignment need their own coordinates. The lesson gap $0.82-0.49=0.330$ retains $0.49/0.82=0.598$; helpfulness, honesty, and safety average to $0.733$ but the required-floor score is $0.400$.

In [ ]:
gap = 0.82 - 0.49
retained = 0.49 / 0.82
before = np.array([0.85, 0.80, 0.78])
after = np.array([0.55, 0.57, 0.90])
old_change = np.mean(after[:2] - before[:2])
alignment = np.array([0.92, 0.88, 0.40])
alignment_average = float(np.mean(alignment))
alignment_floor = float(np.min(alignment))
scale_factor = 100 ** (-0.08)
scaled_loss = 0.300 * scale_factor
improvement = 0.300 - scaled_loss

assert round(gap, 3) == 0.330
assert round(retained, 3) == 0.598
assert round(old_change, 3) == -0.265
assert round(alignment_average, 3) == 0.733
assert round(alignment_floor, 3) == 0.400
assert round(scale_factor, 3) == 0.692
assert round(scaled_loss, 3) == 0.208
assert round(improvement, 3) == 0.092

print("retained", round(retained, 3))
print("old task change", round(old_change, 3))
print("alignment average and floor", round(alignment_average, 3), round(alignment_floor, 3))

## The dataset ladder

The F16 evaluation ladder rises from the lesson scorecard to clean multitask scores, shifted scores, continual-learning before/after matrices, and a D5 alignment-floor scenario with a low safety coordinate.

In [ ]:
def make_agi_ladder():
    return [
        {"name": "D1 four capabilities", "caps": np.array([0.9, 0.8, 0.2, 0.4]), "shift": None, "floors": None, "complexity": 1},
        {"name": "D2 clean benchmark", "caps": np.array([0.82, 0.77, 0.74, 0.70, 0.68]), "shift": None, "floors": None, "complexity": 2},
        {"name": "D3 shifted eval", "caps": np.array([0.82, 0.80, 0.78, 0.75]), "shift": np.array([0.49, 0.55, 0.46, 0.50]), "floors": None, "complexity": 4},
        {"name": "D4 continual review", "caps": np.array([0.55, 0.57, 0.90, 0.62]), "shift": np.array([0.50, 0.52, 0.78, 0.58]), "floors": None, "complexity": 6},
        {"name": "D5 alignment floor", "caps": np.array([0.92, 0.88, 0.40, 0.76]), "shift": np.array([0.80, 0.70, 0.35, 0.68]), "floors": np.array([0.70, 0.70, 0.80, 0.60]), "complexity": 10},
    ]

agi_ladder = make_agi_ladder()

for rung in agi_ladder:
    print(rung["name"], "shape", rung["caps"].shape, "complexity", rung["complexity"], "sample", np.round(rung["caps"], 3))

In [ ]:
agi_results = []

for rung in agi_ladder:
    result = agi_score(rung["caps"], shift_scores=rung["shift"], floors=rung["floors"])
    agi_results.append(result)
    print(f"{rung['name']}: average={result['average']:.3f} bottleneck={result['bottleneck']:.3f} retained={result['retained']:.3f} robust={result['robust']:.3f}")

print("single-average baseline on D5", round(agi_results[-1]["average"], 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))

for i, rung in enumerate(agi_ladder):
    ax = axes[0, i]
    coords = rung["caps"]
    ax.bar(np.arange(len(coords)), coords)
    ax.axhline(agi_results[i]["bottleneck"], color="red", linestyle="--")
    ax.set_ylim(0, 1)
    ax.set_title(rung["name"])
    ax.set_xlabel("coordinate")

complexity = [rung["complexity"] for rung in agi_ladder]
robust = [result["robust"] for result in agi_results]
axes[1, 0].plot(complexity, robust, marker="o")
axes[1, 0].set_ylim(0, 1)
axes[1, 0].set_title("Bottleneck-aware score")
axes[1, 0].set_xlabel("evaluation complexity")

for j in range(1, 5):
    axes[1, j].bar(["avg", "geo", "min"], [agi_results[j]["average"], agi_results[j]["geometric"], agi_results[j]["bottleneck"]])
    axes[1, j].set_ylim(0, 1)
    axes[1, j].set_title(f"D{j + 1} score views")

plt.tight_layout()

## Pitfall on the hardest rung: treating alignment as a postscript

D5 looks strong by average, but the safety coordinate is below the required floor. Hard gating prevents helpfulness or fluency from averaging safety away.

In [ ]:
d5 = agi_ladder[-1]
wrong_average = float(np.mean(d5["caps"][:3]))
wrong_floor = float(np.min(d5["caps"][:3]))
gated = agi_score(d5["caps"], shift_scores=d5["shift"], floors=d5["floors"])
fixed_deployment_score = gated["robust"] if np.all(d5["caps"] >= d5["floors"] * 0.5) else 0.0

assert round(wrong_average, 3) == 0.733
assert round(wrong_floor, 3) == 0.400

print("unsafe average", round(wrong_average, 3))
print("required-floor score", round(wrong_floor, 3))
print("gated robust score", round(gated["robust"], 3))
print("deployment score after hard gate", round(fixed_deployment_score, 3))

## Evaluate it + Practice
- Compare the rung metric against the no-skill baseline printed above.
- Run one cheap sanity check: change one label, threshold, route, or floor and confirm the metric moves in the expected direction.
- Ablate the key idea by turning off the kernel, leak, tool verification, feedback, or safety gate.
- Watch failure signals: flat metric curves, hidden weak coordinates, high cost, or a D5 pitfall that does not improve after the fix.

Practice prompts:
1. Change the D3 noise or shift level and predict which rung changes most.

2. Replace the baseline with a stronger classical or rule-based check and compare the metric.

3. Add one new D5 stress case and explain whether the pitfall fix still works.